# Local Whisper no Google Colab

Gera legendas `.srt` em português a partir de vídeos em qualquer idioma no Google Drive, em duas fases na GPU: o [faster-whisper](https://github.com/SYSTRAN/faster-whisper) transcreve todos os vídeos no idioma original e, depois de descarregado, o [Hunyuan-MT-7B](https://huggingface.co/tencent/Hunyuan-MT-7B) traduz os arquivos gerados.

**Antes de rodar:** `Ambiente de execução` → `Alterar tipo de ambiente de execução` → **T4 GPU**.

Fluxo:
1. Verificar GPU
2. Instalar dependências
3. Montar o Google Drive
4. Configurar pasta de entrada, profundidade da varredura e modelos
5. Definir as funções auxiliares
6. Transcrever todos os vídeos e guardar a lista de arquivos gerados
7. Descarregar o Whisper, carregar o tradutor e traduzir os arquivos da lista

Para `video.mkv` em japonês, são gerados `video.ja.srt` (original) e `video.srt` (tradução). Vídeos em português não passam pela tradução.

> O idioma é detectado por vídeo, não por trecho. A licença do Hunyuan-MT não se aplica à União Europeia, ao Reino Unido nem à Coreia do Sul.

## 1. Verificar GPU

Se `nvidia-smi` falhar, o ambiente não está com GPU — troque o runtime antes de continuar.

In [ ]:
!nvidia-smi

## 2. Instalar dependências

`av<14` reproduz a pin do projeto local. `transformers`, `accelerate` e `bitsandbytes` rodam o tradutor em 4 bits. O `ctranslate2` precisa de CUDA 12 e cuDNN 9, que o runtime do Colab pode não trazer. Por isso a célula instala essas bibliotecas via pip e as registra no sistema com `ldconfig`.

In [ ]:
!pip install -q "faster-whisper>=1.2.1" "av<14" "transformers>=4.56" accelerate bitsandbytes nvidia-cublas-cu12 "nvidia-cudnn-cu12==9.*"

import site
import subprocess
from pathlib import Path

# LD_LIBRARY_PATH só vale no início do processo; o ldconfig funciona com o kernel já rodando.
cuda_lib_dirs = set()
for packages_dir in site.getsitepackages():
    for lib_name in ("libcublas.so.12", "libcudnn.so.9"):
        for lib in Path(packages_dir).glob(f"nvidia/**/{lib_name}"):
            cuda_lib_dirs.add(str(lib.parent))

if not cuda_lib_dirs:
    raise RuntimeError("libcublas.so.12 e libcudnn.so.9 não encontradas após o pip install")

Path("/etc/ld.so.conf.d/cuda12-pip.conf").write_text("\n".join(sorted(cuda_lib_dirs)) + "\n")
subprocess.run(["ldconfig"], check=True)

ldconfig_output = subprocess.run(["ldconfig", "-p"], capture_output=True, text=True).stdout
for lib_name in ("libcublas.so.12", "libcudnn.so.9"):
    status = "ok" if lib_name in ldconfig_output else "NÃO ENCONTRADA"
    print(f"[*] {lib_name}: {status}")

## 3. Montar o Google Drive

Os vídeos devem estar em alguma pasta do seu Drive. Cada legenda será salva com o mesmo nome do vídeo (`video.mp4` → `video.srt`).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. Configuração

- **INPUT_PATH**: pasta ou arquivo de vídeo dentro do Drive.
- **SCAN_DEPTH**: quantos níveis de subpastas varrer abaixo de `INPUT_PATH`. `0` = só a própria pasta; `1` = filmes (`filmes/Nome do Filme/filme.mkv`); `2` = séries (`series/Nome da Série/Season 01/ep.mkv`).
- **OUTPUT_PATH**: pasta de saída, que espelha as subpastas de `INPUT_PATH` (vazio = salvar ao lado do vídeo).
- **MODEL_SIZE**: `large-v3` aproveita o T4 e dá o melhor resultado.
- **SOURCE_LANGUAGE**: idioma do áudio (`ja`, `en`...). Vazio = detectar automaticamente. Preencha se a detecção errar, ou ao rodar uma pasta de um idioma só.
- **TRANSLATION_MODEL**: modelo de tradução do Hugging Face, carregado em 4 bits. O primeiro uso baixa cerca de 16 GB.
- **TRANSLATION_BATCH_SIZE**: legendas traduzidas por vez. Reduza se faltar VRAM.
- **FORCE**: `True` reprocessa mesmo que já exista uma legenda.

In [ ]:
INPUT_PATH = "/content/drive/MyDrive/videos"  # @param {type:"string"}
SCAN_DEPTH = 2  # @param {type:"integer"}
OUTPUT_PATH = ""  # @param {type:"string"}
MODEL_SIZE = "large-v3"  # @param ["tiny", "base", "small", "medium", "large-v3"]
SOURCE_LANGUAGE = ""  # @param {type:"string"}
TRANSLATION_MODEL = "tencent/Hunyuan-MT-7B"  # @param {type:"string"}
TRANSLATION_BATCH_SIZE = 8  # @param {type:"integer"}
BEAM_SIZE = 5  # @param {type:"integer"}
FORCE = False  # @param {type:"boolean"}

## 5. Funções auxiliares

Adaptação do pacote `local_whisper` (`files.py`, `srt.py`, `transcriber.py`) para rodar em um único notebook, mais a etapa de tradução.

In [ ]:
from __future__ import annotations

import gc
import os
import re
import time
from pathlib import Path
from typing import NamedTuple

import torch
from tqdm.auto import tqdm

VIDEO_EXTENSIONS = {".mp4", ".mkv", ".avi", ".mov", ".webm", ".flv", ".wmv", ".m4v", ".ts"}

SUBTITLE_EXTENSIONS = {
    ".srt", ".vtt", ".ass", ".ssa", ".sub", ".sbv",
    ".ttml", ".dfxp", ".smi", ".stl", ".usf", ".idx",
}

TARGET_LANGUAGE = "pt"
TARGET_LANGUAGE_NAME = "Brazilian Portuguese"
TRANSLATION_PROMPT = "Translate the following segment into {target}, without additional explanation.\n\n{text}"
TRANSLATION_MAX_NEW_TOKENS = 256
LANGUAGE_DETECTION_SEGMENTS = 5

SRT_BLOCK_SEPARATOR = re.compile(r"\n\s*\n")


class TranslationJob(NamedTuple):
    source: Path
    dest: Path


class Cue(NamedTuple):
    timing: str
    text: str


def collect_videos(path: str, depth: int) -> list[Path]:
    root_path = Path(path)
    if root_path.is_file():
        return [root_path] if root_path.suffix.lower() in VIDEO_EXTENSIONS else []

    videos: list[Path] = []
    for current_dir, subdirs, filenames in os.walk(root_path):
        current_path = Path(current_dir)
        level = len(current_path.relative_to(root_path).parts)
        if level >= depth:
            subdirs.clear()
        for filename in filenames:
            file_path = current_path / filename
            if file_path.suffix.lower() in VIDEO_EXTENSIONS:
                videos.append(file_path)
    return sorted(videos)


def existing_subtitle(video: Path, extra_dir: Path | None = None) -> Path | None:
    dirs = [video.parent]
    if extra_dir and extra_dir != video.parent:
        dirs.append(extra_dir)
    for d in dirs:
        for ext in SUBTITLE_EXTENSIONS:
            candidate = d / (video.stem + ext)
            if candidate.exists():
                return candidate
    return None


def format_timestamp(seconds: float) -> str:
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = int(seconds % 60)
    ms = int((seconds - int(seconds)) * 1000)
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"


def segments_to_cues(segments: list) -> list[Cue]:
    return [
        Cue(f"{format_timestamp(seg.start)} --> {format_timestamp(seg.end)}", seg.text.strip())
        for seg in segments
    ]


def write_srt(dest: Path, cues: list[Cue]) -> None:
    lines: list[str] = []
    for i, cue in enumerate(cues, start=1):
        lines.extend([str(i), cue.timing, cue.text, ""])
    dest.parent.mkdir(parents=True, exist_ok=True)
    dest.write_text("\n".join(lines), encoding="utf-8")


def read_srt(path: Path) -> list[Cue]:
    cues: list[Cue] = []
    for block in SRT_BLOCK_SEPARATOR.split(path.read_text(encoding="utf-8").strip()):
        lines = block.splitlines()
        if len(lines) >= 3:
            cues.append(Cue(lines[1], " ".join(lines[2:])))
    return cues


def transcribe_file(model, video: Path, output_dir: Path | None,
                    index: int = 0, total: int = 0,
                    source_language: str = "", beam_size: int = 5) -> TranslationJob | None:
    base_dir = output_dir or video.parent
    dest = base_dir / (video.stem + ".srt")
    batch = f" ({index}/{total})" if total else ""
    print(f"  → {video.name}{batch}")

    start = time.perf_counter()
    segments_gen, info = model.transcribe(
        str(video),
        language=source_language or None,
        language_detection_segments=LANGUAGE_DETECTION_SEGMENTS,
        beam_size=beam_size,
        vad_filter=True,
        vad_parameters=dict(min_silence_duration_ms=500),
    )
    print(f"    idioma: {info.language} (confiança {info.language_probability:.2f})")

    duration = info.duration
    collected = []
    bar = tqdm(
        total=int(duration),
        unit="s",
        bar_format="    {l_bar}{bar}| {n_fmt}/{total_fmt}s [{elapsed}<{remaining}]",
        leave=False,
    )
    prev_end = 0.0
    for seg in segments_gen:
        collected.append(seg)
        delta = seg.end - prev_end
        if delta > 0:
            bar.update(int(delta))
            prev_end = seg.end
    bar.update(bar.total - bar.n)
    bar.close()

    elapsed = time.perf_counter() - start

    if not collected:
        print(f"    ⚠ nenhuma fala detectada ({elapsed:.1f}s)")
        return None

    cues = segments_to_cues(collected)

    if info.language == TARGET_LANGUAGE:
        write_srt(dest, cues)
        print(f"    ✓ {len(cues)} segmentos — {elapsed:.1f}s → {dest.name}")
        return None

    original_dest = base_dir / f"{video.stem}.{info.language}.srt"
    write_srt(original_dest, cues)
    print(f"    ✓ {len(cues)} segmentos — {elapsed:.1f}s → {original_dest.name}")
    return TranslationJob(original_dest, dest)


def build_translation_prompt(tokenizer, text: str) -> str:
    content = TRANSLATION_PROMPT.format(target=TARGET_LANGUAGE_NAME, text=text)
    messages = [{"role": "user", "content": content}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)


def translate_batch(translator, tokenizer, texts: list[str]) -> list[str]:
    prompts = [build_translation_prompt(tokenizer, text) for text in texts]
    inputs = tokenizer(prompts, return_tensors="pt", padding=True, add_special_tokens=False)
    inputs = inputs.to(translator.device)
    with torch.inference_mode():
        output_tokens = translator.generate(**inputs, max_new_tokens=TRANSLATION_MAX_NEW_TOKENS)
    prompt_length = inputs["input_ids"].shape[1]
    new_tokens = output_tokens[:, prompt_length:]
    decoded = tokenizer.batch_decode(new_tokens, skip_special_tokens=True)
    return [text.strip() for text in decoded]


def translate_file(translator, tokenizer, job: TranslationJob, batch_size: int) -> None:
    print(f"  → {job.source.name}")
    start = time.perf_counter()
    cues = read_srt(job.source)

    translated: list[Cue] = []
    for i in tqdm(range(0, len(cues), batch_size), unit="lote", leave=False):
        batch = cues[i:i + batch_size]
        texts = translate_batch(translator, tokenizer, [cue.text for cue in batch])
        translated.extend(Cue(cue.timing, text) for cue, text in zip(batch, texts))

    write_srt(job.dest, translated)
    elapsed = time.perf_counter() - start
    print(f"    ✓ {len(translated)} legendas — {elapsed:.1f}s → {job.dest.name}")

## 6. Transcrever todos os vídeos

Usa `float16` na GPU (rápido e enxuto na VRAM do T4) ou `int8` como fallback de CPU. Pula vídeos que já têm legenda (a menos que `FORCE=True`), e erros em um arquivo não interrompem o lote. Cada `.srt` gerado que precisa de tradução entra na lista `translation_jobs`, usada na próxima etapa.

In [ ]:
import ctranslate2
from faster_whisper import WhisperModel

videos = collect_videos(INPUT_PATH, SCAN_DEPTH)
if not videos:
    raise SystemExit(f"Nenhum vídeo encontrado em {INPUT_PATH} (SCAN_DEPTH={SCAN_DEPTH})")

input_path = Path(INPUT_PATH)
input_root = input_path if input_path.is_dir() else input_path.parent
output_root = Path(OUTPUT_PATH) if OUTPUT_PATH else None


def output_dir_for(video: Path) -> Path | None:
    if output_root is None:
        return None
    return output_root / video.parent.relative_to(input_root)


if not FORCE:
    filtered = []
    for v in videos:
        sub = existing_subtitle(v, output_dir_for(v))
        if sub:
            print(f"[=] pulando {v.name} — já tem legenda ({sub.name})")
        else:
            filtered.append(v)
    videos = filtered

translation_jobs: list[TranslationJob] = []

if not videos:
    print("\n[✓] Nada a fazer — todos os vídeos já têm legenda")
else:
    if ctranslate2.get_cuda_device_count() > 0:
        device, compute_type = "cuda", "float16"
    else:
        device, compute_type = "cpu", "int8"
        print("[!] GPU não detectada — rodando em CPU (vai ser lento). Troque o runtime para T4.")

    model = WhisperModel(MODEL_SIZE, device=device, compute_type=compute_type)
    print(f"[*] Whisper: {MODEL_SIZE} | Dispositivo: {device} ({compute_type})")
    print(f"[*] {len(videos)} vídeo(s) para transcrever\n")

    for i, video in enumerate(videos, start=1):
        try:
            job = transcribe_file(model, video, output_dir_for(video),
                                  index=i, total=len(videos),
                                  source_language=SOURCE_LANGUAGE, beam_size=BEAM_SIZE)
            if job:
                translation_jobs.append(job)
        except Exception as e:
            print(f"    ✗ erro em {video.name}: {e}")

    print(f"\n[✓] Transcrição concluída — {len(translation_jobs)} arquivo(s) para traduzir")

## 7. Traduzir os arquivos gerados

Descarrega o Whisper para liberar a VRAM, carrega o tradutor em 4 bits e traduz os arquivos de `translation_jobs`. O tradutor precisa de GPU. Arquivos traduzidos saem da lista, então, se algum falhar, basta rodar esta célula de novo para tentar só os que faltam, sem repetir a transcrição.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model = None
gc.collect()
torch.cuda.empty_cache()

if not translation_jobs:
    print("[✓] Nada para traduzir")
else:
    if not torch.cuda.is_available():
        raise RuntimeError("O tradutor em 4 bits precisa de GPU. Troque o runtime para T4.")

    if "translator" not in globals():
        quantization = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
        )
        tokenizer = AutoTokenizer.from_pretrained(TRANSLATION_MODEL, padding_side="left")
        translator = AutoModelForCausalLM.from_pretrained(
            TRANSLATION_MODEL,
            quantization_config=quantization,
            device_map="auto",
        )
    print(f"[*] Tradutor: {TRANSLATION_MODEL} | VRAM: {torch.cuda.memory_allocated() / 1e9:.1f} GB")
    print(f"[*] {len(translation_jobs)} arquivo(s) para traduzir\n")

    for job in list(translation_jobs):
        try:
            translate_file(translator, tokenizer, job, TRANSLATION_BATCH_SIZE)
            translation_jobs.remove(job)
        except Exception as e:
            print(f"    ✗ erro em {job.source.name}: {e}")

    if translation_jobs:
        print(f"\n[!] {len(translation_jobs)} arquivo(s) não traduzido(s) — rode esta célula de novo")
    else:
        print("\n[✓] Concluído")